# Gemma 4 26B-A4B

Attach the uploaded `kaggle_new_models` dataset, enable Internet, and select **T4 x2** manually. This notebook runs only `gemma4` with expert-NF4 precision. Experiments 1 and 2; TRC uses eight separately encoded frames. See README.md for methodology and limits.


In [ ]:
from pathlib import Path
import subprocess, sys, os, shutil
matches = list(Path('/kaggle/input').rglob('kaggle_new_models/run.py'))
assert len(matches) == 1, f'Expected one uploaded package; found {matches}'
CODE = matches[0].parent
MODEL = 'gemma4'
OUTPUT = Path('/kaggle/working/gemma4_vision_fp32_results')
EXPERIMENTS = [1, 2]
SAMPLES = 256  # Set to 1000 before starting if you also want DeepSeek's sample count.
RUN_ARGS = ['--output-root', str(OUTPUT), '--experiments', *map(str, EXPERIMENTS)]
MODEL_ARGS = []
os.environ['HF_XET_CHUNK_CACHE_SIZE_BYTES'] = '0'
os.environ['HF_HUB_DISABLE_XET'] = '1'
os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '0'
os.environ.update(USE_TORCH='1', USE_TF='0', USE_FLAX='0', USE_TORCH_XLA='0', PYTEST_DISABLE_PLUGIN_AUTOLOAD='1', TOKENIZERS_PARALLELISM='false')
print('Notebook version: 2026-09-30-gemma4-vision-fp32')
assert '2026-09-30-gemma4-vision-fp32' in (CODE / 'gemma_loading.py').read_text(), 'Update the attached Kaggle code package.'
subprocess.run([sys.executable, '-u', str(CODE / 'setup_runtime.py'), '--model', MODEL], check=True)


## Optional development tests
Leave `RUN_DEVELOPMENT_TESTS = False` for the experiment run. The actual model smoke test below checks unpruned native logits and all selected experiment conditions.


In [ ]:
RUN_DEVELOPMENT_TESTS = False
if RUN_DEVELOPMENT_TESTS:
    subprocess.run([sys.executable, '-m', 'pytest', str(CODE / 'test_gemma.py'), str(CODE / 'test_core.py'), '-q', '-p', 'no:cacheprovider'], check=True)
else:
    print('Development tests skipped; proceed to the model smoke test.')


## Small CUDA preflight
Run these tiny-model checks before downloading the full checkpoint. They exercise the FP32 vision/FP16 text boundary, expert NF4 and GPU dispatch. This does not replace the real-model smoke test.


In [ ]:
subprocess.run([sys.executable, '-m', 'pytest', str(CODE / 'test_gemma.py'), '-q', '-p', 'no:cacheprovider', '-k', 'full_loading_path or fp32_vision or nf4_experts or nonfinite_trace'], env={**os.environ, 'OMP_NUM_THREADS': '2', 'MKL_NUM_THREADS': '2'}, check=True)


## Smoke test
Two examples per selected experiment, in separate folders. Experiment 1 runs baseline, L2 at 50% retention, and random at 50% retention. Gemma weights are streamed in 32 MiB HTTP windows; its 50 GB shard is never saved to working disk. Vision runs in FP32, one image at a time; all eight video frames are retained. Diagnostics: `/kaggle/working/model_shards/gemma4_loading_memory.jsonl` and `gemma4_numerics.jsonl` (written if invalid values are found). A failed native-equivalence check or incomplete matrix stops the subprocess.


In [ ]:
subprocess.run([sys.executable, '-u', str(CODE / 'run.py'), '--model', MODEL, '--smoke', *RUN_ARGS, *MODEL_ARGS], check=True)
subprocess.run([sys.executable, str(CODE / 'analyze.py'), '--model', MODEL, '--smoke', *RUN_ARGS], check=True)


In [ ]:
import json, pandas as pd
for experiment in EXPERIMENTS:
    root = OUTPUT / MODEL / f'experiment_{experiment}_smoke'
    print(f'Experiment {experiment}:', json.loads((root / 'native_equivalence.json').read_text()))
    display(pd.read_csv(root / 'summaries' / 'summary.csv'))


## Full runs
After inspecting the smoke results, run this cell. Each selected experiment uses `SAMPLES` examples (default 256). Experiment 1 runs baseline, L2 at 50% retention, and random at 50% retention. Experiment 2 runs all four frame conditions. Each condition uses the same sampled examples within the run. The sweep has its own output root; do not merge the old 50%-only folders into it. Rerun the same cell to resume with the same configuration; results must still be present under `/kaggle/working`. Model loading downloads shards again because the smoke subprocess released its model. Full runtime has not been measured; save outputs before the Kaggle session expires.


In [ ]:
subprocess.run([sys.executable, '-u', str(CODE / 'run.py'), '--model', MODEL, '--limit', str(SAMPLES), *RUN_ARGS, *MODEL_ARGS], check=True)
subprocess.run([sys.executable, str(CODE / 'analyze.py'), '--model', MODEL, '--limit', str(SAMPLES), *RUN_ARGS], check=True)


In [ ]:
for experiment in EXPERIMENTS:
    root = OUTPUT / MODEL / f'experiment_{experiment}_n{SAMPLES}'
    summary = pd.read_csv(root / 'summaries' / 'summary.csv')
    display(summary)
    if experiment == 1:
        columns = ['condition', 'examples', 'target_retention', 'actual_retention', 'accuracy', 'accuracy_delta', 'racr_example', 'top1_flip', 'accuracy_delta_ci95_low', 'accuracy_delta_ci95_high']
        print(summary[columns].to_string(index=False))
if 2 in EXPERIMENTS:
    display(pd.read_csv(OUTPUT / MODEL / f'experiment_2_n{SAMPLES}/summaries/condition_accuracy.csv'))


## Download outputs
This cell can also be run after an interrupted experiment to preserve partial progress. Keep raw routes, predictions, manifests and configuration with the summaries.


In [ ]:
diagnostics = OUTPUT / 'diagnostics'
diagnostics.mkdir(parents=True, exist_ok=True)
for log in Path('/kaggle/working/model_shards').glob('*.jsonl'):
    shutil.copy2(log, diagnostics / log.name)
archive = shutil.make_archive(f'/kaggle/working/{MODEL}_experiment_results', 'zip', OUTPUT)
from IPython.display import FileLink, display
display(FileLink(archive))
